# MediGuide — Prompt Tuning
### Qwen2.5-1.5B-Instruct + PEFT Prompt Tuning on Doctor-Patient Conversations

Unlike LoRA/QLoRA, **the entire base model stays frozen** — 100% of its ~1.5B parameters are
untouched. The only trainable parameters are a small set of "virtual token" embeddings
(`num_virtual_tokens=20`, same 1536-dim as the model's real token embeddings — about 30K
trainable params total, vs LoRA's ~9M). These get prepended to every input sequence; the model
learns to steer its own frozen behavior purely by what those virtual tokens encode.

**Expectation going in, stated plainly so the numbers aren't a surprise:** prompt tuning can only
bias what the frozen model already tends to do — it can't reshape internal representations the
way LoRA/QLoRA can. Given that LoRA's main win over the zero-shot baseline was *reshaping output
length/register* (the baseline was stuck outputting a near-fixed ~190-word answer regardless of
the question; LoRA pulled that down to properly variable lengths matching the reference), I'd
expect prompt tuning to underperform both LoRA and QLoRA on ROUGE/BLEU/perplexity here — that's
a genuine content-shaping problem, and 20 soft tokens have limited leverage over it. That's not
a reason to skip this run — it's the right data point for the report's cost/quality trade-off
table (spec explicitly asks for it), just don't expect it to compete with LoRA/QLoRA on quality.

Same seed (8), same tokenizer (unmodified), same prompt template, same eval pipeline as the
previous three notebooks.


## 1. Setup

In [ ]:
!pip install -q -U peft accelerate bitsandbytes evaluate rouge_score sacrebleu sentencepiece
!pip uninstall -y torchao -q


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single GPU only — avoids Trainer's automatic
                                            # DataParallel wrapping, which caused issues before
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # must be set before torch initializes CUDA

import json, time, math, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq,
)
from peft import PromptTuningConfig, PromptTuningInit, get_peft_model, TaskType

SEED = 8
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
DATA_DIR = "/kaggle/input/datasets/totaldose/doctor-patient-conversation/data/processed"

def find_split_file(data_dir, split_names):
    for name in split_names:
        p = os.path.join(data_dir, f"{name}.json")
        if os.path.exists(p):
            return p
    return None

TRAIN_PATH = find_split_file(DATA_DIR, ["train"])
VAL_PATH   = find_split_file(DATA_DIR, ["val", "valid", "validation", "dev"])
TEST_PATH  = find_split_file(DATA_DIR, ["test"])

print("train:", TRAIN_PATH)
print("val:  ", VAL_PATH)
print("test: ", TEST_PATH)
assert TRAIN_PATH is not None and TEST_PATH is not None, "Check DATA_DIR"


In [ ]:
# Prior results, for the final 4-way comparison table
RESULT_CANDIDATES = {
    "baseline": ["/kaggle/working/results/baseline_summary.json",
                 "/kaggle/input/mediguide-baseline/baseline_summary.json"],
    "lora": ["/kaggle/working/results/lora_summary.json",
             "/kaggle/input/mediguide-lora/lora_summary.json"],
    "qlora": ["/kaggle/working/results/qlora_summary.json",
              "/kaggle/input/mediguide-qlora/qlora_summary.json"],
}
prior_summaries = {}
for name, candidates in RESULT_CANDIDATES.items():
    path = next((p for p in candidates if os.path.exists(p)), None)
    if path:
        with open(path) as f:
            prior_summaries[name] = json.load(f)
        print(f"Found {name} summary at {path}")
    else:
        print(f"WARNING: {name}_summary.json not found automatically — set its path manually "
              f"below if you want it in the final comparison table.")


## 2. Tokenizer — unmodified, sanity-checked

Same check as every previous notebook. Prompt tuning doesn't touch the tokenizer at all — it
adds trainable *embedding vectors*, not new vocabulary tokens, so this check is really just
consistency hygiene at this point, not a specific risk for this method.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME)

print("len(tokenizer):    ", len(tokenizer))
print("config.vocab_size: ", config.vocab_size)
assert config.vocab_size >= len(tokenizer)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # right-padding for training


## 3. Load base model (frozen) + apply Prompt Tuning

We initialize the virtual tokens from real text (`prompt_tuning_init="TEXT"`) rather than random
noise — this seeds them with a sensible starting point in embedding space and generally converges
faster/more reliably than random init for this few trainable parameters.

Gradient checkpointing stays on (same OOM lesson from the LoRA/QLoRA runs) — even though no base
weights update, the backward pass still needs to keep the frozen model's activations around to
propagate gradients back to the virtual tokens, so the memory pressure is closer to full
fine-tuning's forward/backward than to LoRA's.


In [ ]:
torch.cuda.reset_peak_memory_stats()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16
).to(DEVICE)

base_model_mem_gb = torch.cuda.memory_allocated() / (1024**3)
print(f"GPU memory after loading fp16 base model: {base_model_mem_gb:.3f} GB")

base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False
base_model.enable_input_require_grads()  # ensures gradient checkpointing can reach the virtual
                                          # token embeddings correctly during backward


In [ ]:
NUM_VIRTUAL_TOKENS = 20

prompt_tuning_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=NUM_VIRTUAL_TOKENS,
    prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text=(
        "Answer the patient's medical question professionally, following recognized "
        "clinical guidelines, and include an appropriate informational-only disclaimer."
    ),
    tokenizer_name_or_path=MODEL_NAME,
)

model = get_peft_model(base_model, prompt_tuning_config)
model.print_trainable_parameters()


## 4. Dataset preparation (identical to the LoRA/QLoRA notebooks)

No changes needed here — `PeftModelForCausalLM` (what `get_peft_model` returns for prompt tuning)
automatically prepends the virtual tokens to `input_ids`/`inputs_embeds` and pads `labels`/
`attention_mask` accordingly at the start of its own `forward()`. Our existing
`input_ids`/`labels`/`attention_mask` encoding works unmodified.


In [ ]:
SYSTEM_PROMPT = (
    "You are a medical information assistant. Respond to the patient's question with "
    "clear, professional, and clinically sound guidance, consistent with recognized clinical "
    "guidelines. Use formal medical language appropriate for a patient audience. Always make "
    "clear that your response is informational only and does not replace an in-person "
    "diagnosis or professional medical care."
)

MAX_PROMPT_LENGTH = 768
MAX_TARGET_LENGTH = 384
MAX_TOTAL_LENGTH = MAX_PROMPT_LENGTH + MAX_TARGET_LENGTH

def build_prompt(description, patient, for_generation=True):
    user_turn = f"{description.strip()}\n\n{patient.strip()}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_turn},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=for_generation
    )

def encode_example(description, patient, doctor):
    prompt_text = build_prompt(description, patient, for_generation=True)
    prompt_ids = tokenizer(
        prompt_text, truncation=True, max_length=MAX_PROMPT_LENGTH, add_special_tokens=False
    )["input_ids"]
    target_ids = tokenizer(
        doctor.strip() + tokenizer.eos_token,
        truncation=True, max_length=MAX_TARGET_LENGTH, add_special_tokens=False
    )["input_ids"]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}


In [ ]:
class DoctorPatientDataset(Dataset):
    def __init__(self, df):
        self.rows = df.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        return encode_example(row["Description"], row["Patient"], row["Doctor"])


def load_split(path):
    with open(path) as f:
        return pd.DataFrame(json.load(f))

df_train = load_split(TRAIN_PATH)
df_val = load_split(VAL_PATH) if VAL_PATH else None
df_test = load_split(TEST_PATH)

print("train:", len(df_train))
if df_val is not None:
    print("val:  ", len(df_val))
print("test: ", len(df_test))

train_dataset = DoctorPatientDataset(df_train)
eval_dataset = DoctorPatientDataset(df_val) if df_val is not None else None

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100,
)


## 5. Training configuration

**Learning rate is much higher than LoRA/QLoRA's `2e-4`.** This isn't a typo — soft-prompt
embeddings are optimized directly with no architectural reparameterization damping the gradient
(unlike LoRA's low-rank decomposition), so prompt tuning conventionally needs a noticeably larger
LR to converge in a comparable number of steps. `3e-2` is the PEFT library's own reference default
for this method. Same 3 epochs, same effective batch size (16) as the other runs for a fair
comparison, but this is worth revisiting if the loss curve looks like it hasn't converged.


In [ ]:
OUTPUT_DIR = "/kaggle/working/prompt_tuning_checkpoints"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,   # effective batch size 16, matches LoRA/QLoRA runs
    learning_rate=3e-2,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,                 # prompt tuning conventionally skips weight decay on the soft prompt
    fp16=True,
    logging_steps=20,
    eval_strategy="epoch" if eval_dataset is not None else "no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=eval_dataset is not None,
    metric_for_best_model="eval_loss" if eval_dataset is not None else None,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)


In [ ]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()

train_result = trainer.train()

train_wall_seconds = time.time() - t0
peak_train_mem_gb = torch.cuda.max_memory_allocated() / (1024**3)

print(train_result)
print(f"\nTraining wall-clock time: {train_wall_seconds:.1f}s")
print(f"Peak GPU memory during training: {peak_train_mem_gb:.3f} GB")


## 6. Save the adapter

This should be tiny — a few hundred KB, not tens of MB like LoRA/QLoRA — since it's just
`num_virtual_tokens × hidden_size` floats (20 × 1536 here) and nothing else.


In [ ]:
ADAPTER_DIR = "/kaggle/working/prompt_tuning_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Adapter directory contents:", os.listdir(ADAPTER_DIR))

adapter_file = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
if os.path.exists(adapter_file):
    adapter_size_mb = os.path.getsize(adapter_file) / (1024 * 1024)
    print(f"Adapter size: {adapter_size_mb:.4f} MB")
else:
    adapter_size_mb = None
    print("adapter_model.safetensors not found — check the contents printed above.")


## 7. Re-run the same evaluation pipeline

Identical to the previous three notebooks: greedy decoding, same `MAX_NEW_TOKENS`, teacher-forced
perplexity, ROUGE-L/BLEU. `model.generate()` on a `PeftModelForCausalLM` for prompt tuning
automatically prepends the trained virtual tokens — no extra code needed here.


In [ ]:
model.eval()
model.config.use_cache = True
tokenizer.padding_side = "left"  # required for batched generation

GEN_MAX_NEW_TOKENS = 256
GEN_BATCH_SIZE = 8

def generate_batch(prompts, max_new_tokens=GEN_MAX_NEW_TOKENS):
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True,
        max_length=MAX_PROMPT_LENGTH
    ).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = time.time() - t0
    new_tokens = out[:, enc["input_ids"].shape[1]:]
    texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return [t.strip() for t in texts], elapsed / len(prompts)


In [ ]:
prompts = [build_prompt(row["Description"], row["Patient"]) for _, row in df_test.iterrows()]

predictions, gen_seconds = [], []
t0 = time.time()
for i in range(0, len(prompts), GEN_BATCH_SIZE):
    batch = prompts[i:i + GEN_BATCH_SIZE]
    texts, per_ex_time = generate_batch(batch)
    predictions.extend(texts)
    gen_seconds.extend([per_ex_time] * len(texts))
    print(f"{min(i + GEN_BATCH_SIZE, len(prompts))}/{len(prompts)} done, "
          f"{time.time() - t0:.1f}s elapsed", flush=True)

df_test["prediction"] = predictions
df_test["gen_seconds"] = gen_seconds
total_gen_time = time.time() - t0
print(f"Total generation time: {total_gen_time:.1f}s")


In [ ]:
def compute_example_loss(prompt, target):
    prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_PROMPT_LENGTH,
                            add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target + tokenizer.eos_token, add_special_tokens=False,
                            truncation=True, max_length=MAX_TARGET_LENGTH)["input_ids"]
    input_ids = torch.tensor([prompt_ids + target_ids]).to(DEVICE)
    labels = input_ids.clone()
    labels[:, :len(prompt_ids)] = -100
    with torch.no_grad():
        out = model(input_ids=input_ids, labels=labels)
    return out.loss.item(), len(target_ids)


In [ ]:
losses, n_target_tokens = [], []
t0 = time.time()
for i, row in df_test.iterrows():
    prompt = build_prompt(row["Description"], row["Patient"])
    loss, n_tok = compute_example_loss(prompt, row["Doctor"])
    losses.append(loss)
    n_target_tokens.append(n_tok)
    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{len(df_test)} scored, {time.time() - t0:.1f}s elapsed", flush=True)

df_test["loss"] = losses
df_test["n_target_tokens"] = n_target_tokens

total_loss_tokens = (df_test["loss"] * df_test["n_target_tokens"]).sum()
total_tokens = df_test["n_target_tokens"].sum()
corpus_ppl = math.exp(total_loss_tokens / total_tokens)
df_test["perplexity"] = df_test["loss"].apply(math.exp)
print(f"Corpus-level perplexity (Prompt Tuning): {corpus_ppl:.3f}")


In [ ]:
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

references = df_test["Doctor"].tolist()
rouge_scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
bleu_scores = bleu.compute(predictions=predictions, references=[[r] for r in references])

rouge_per_row = [rouge.compute(predictions=[p], references=[r], use_stemmer=True)["rougeL"]
                  for p, r in zip(predictions, references)]
df_test["rougeL"] = rouge_per_row

print("ROUGE:", rouge_scores)
print("BLEU: ", bleu_scores["score"])


## 8. Prompt Tuning results summary + full comparison

In [ ]:
prompt_tuning_summary = {
    "model": MODEL_NAME + " + Prompt Tuning",
    "seed": SEED,
    "num_virtual_tokens": NUM_VIRTUAL_TOKENS,
    "n_test_examples": len(df_test),
    "rouge1": rouge_scores["rouge1"],
    "rouge2": rouge_scores["rouge2"],
    "rougeL": rouge_scores["rougeL"],
    "bleu": bleu_scores["score"],
    "perplexity_corpus": corpus_ppl,
    "avg_generation_seconds_per_example": df_test["gen_seconds"].mean(),
    "max_new_tokens": GEN_MAX_NEW_TOKENS,
    "total_wall_clock_generation_seconds": total_gen_time,
    "adapter_size_mb": adapter_size_mb,
    "peak_training_gpu_mem_gb": peak_train_mem_gb,
    "training_wall_clock_seconds": train_wall_seconds,
}

os.makedirs("/kaggle/working/results", exist_ok=True)
with open("/kaggle/working/results/prompt_tuning_summary.json", "w") as f:
    json.dump(prompt_tuning_summary, f, indent=2)
df_test.to_csv("/kaggle/working/results/prompt_tuning_predictions.csv", index=False)

pd.DataFrame([prompt_tuning_summary]).T.rename(columns={0: "value"})


In [ ]:
# Full 4-way comparison, if the prior three summaries were found earlier
all_summaries = {**prior_summaries, "prompt_tuning": prompt_tuning_summary}
compare_keys = ["rouge1", "rouge2", "rougeL", "bleu", "perplexity_corpus",
                 "avg_generation_seconds_per_example", "adapter_size_mb"]

if len(all_summaries) > 1:
    comparison = pd.DataFrame({
        name: {k: s.get(k) for k in compare_keys} for name, s in all_summaries.items()
    })
    display(comparison.round(4))
else:
    print("Only prompt_tuning_summary available in this session — load the other three "
          "summaries (set their paths in RESULT_CANDIDATES above) for the full table.")


In [ ]:
# Stratified by severity, same as the previous three notebooks
strat = df_test.groupby("Status").agg(
    n=("Doctor", "count"),
    rougeL=("rougeL", "mean"),
    perplexity=("perplexity", "mean"),
).round(4)
strat


In [ ]:
for i in df_test.sample(3, random_state=SEED).index:
    row = df_test.loc[i]
    print("="*100)
    print("SEVERITY:", row["Status"])
    print("PATIENT :", row["Patient"][:250], "...")
    print("-"*100)
    print("REFERENCE DOCTOR:", row["Doctor"][:400])
    print("-"*100)
    print("PROMPT-TUNED PREDICTION:", row["prediction"][:400])


## 9. Next steps

- `results/prompt_tuning_summary.json` and `results/prompt_tuning_predictions.csv` saved.
- `prompt_tuning_adapter/` holds the trained virtual tokens — check its size against the
  expectation of a few hundred KB, versus LoRA/QLoRA's ~70MB.
- With all four runs done (baseline, LoRA, QLoRA, Prompt Tuning), remaining work per the project
  spec: the anonymization/HIPAA-equivalent verification pass, and assembling the final PDF
  performance report (dataset description, the 4-way comparison table, trade-offs discussion,
  and a recommended deployment strategy).
